# Engagement from distance · exp05

Engaged units are the firepower a side actually applies, so they drive the other side's losses.
But a world model can't read future engagement from the data. This notebook tests whether
engagement can be **computed from slower quantities**:

$$E \approx A \cdot s(d)$$

where $A$ is the number of alive units, $d$ the distance between the team centroids and $s(d)$
the share of alive units in range at that distance. It then lets SINDy find that law itself with
a **custom library**.

1. Setup and untrimmed trajectories
2. Measure $s(d)$
3. Fit a smooth step and test the closure
4. Does the closure explain losses?
5. SINDy with a custom library
6. Threshold sweep, and a one-gate library
7. Predicting ahead
8. Engagement as its own state, and why to wait
9. Better geometry: units within reach of an enemy
10. Reach as a state: an equation for it
11. Formation spread

**Setup:** alive and distance enter as **inputs $u$** (SINDYc), read from the data. That isn't a
full world model yet: alive and distance still need equations of their own.

## 1 · Setup

The same loading code and common scale as `walkthrough_exp_05.ipynb`: 1 = ten full-health marines.

Section 9 adds per-unit **geometry** (`geometry`), computed from the same snapshots in the same pass:
`front_distance` (between the closest Blue and Red units) and `reach_*` (alive units with an enemy
within `REACH`, in tens like alive). Section 11 adds `depth_*`: the spread (standard deviation) of each formation's
units along the line between the centroids, in map units.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps
from scipy.optimize import curve_fit

from sindy_wm.io import storage
from sindy_wm.models.gated import fit_gated
from sindy_wm.models.metrics import evaluate_horizons
from sindy_wm.models.trajectories import derivative_data, to_trajectory
from sindy_wm.paths import RAW_DIR
from sindy_wm.states.aggregate import engagement, level_a

pd.set_option("display.width", 160)

DATASET_DIR = RAW_DIR / "exp05_10m_sampled" / "10m_sampled_s2_v1"
N_EPISODES = 5000  # battles to use; None for all 20,000

REF_UNITS, MARINE_HP = 10, 45.0  # common scale: 1 = ten full-health marines
SMOOTH_WINDOW = 9  # Savitzky-Golay window for derivatives, in points

STATE = ["health_blue", "health_red"]  # what SINDy differentiates
INPUTS = ["alive_blue", "alive_red", "distance"]  # given from data, as control inputs u
GEOMETRY = ["front_distance", "reach_blue", "reach_red", "depth_blue", "depth_red"]  # sections 9 and 11
ALL_COLUMNS = STATE + INPUTS + ["engaged_blue", "engaged_red"] + GEOMETRY

REACH = 5 + 2 * 0.375  # marine attack range plus two unit radii: SMAClite's in-range test, centre to centre

In [ ]:
dt = storage.load_manifest(DATASET_DIR)["config"]["env"]["seconds_per_step"]  # 0.125 s
index = storage.load_episode_index(DATASET_DIR).set_index("episode_idx")
ids = list(index.index if N_EPISODES is None else index.index[:N_EPISODES])


def geometry(snapshots):
    # Per step: distance between the closest Blue and Red units, how many alive units have an enemy
    # within REACH, and each formation's depth: the spread of its units along the line between the
    # two centroids
    alive = snapshots[snapshots["alive"]]
    steps = pd.Index(sorted(snapshots["step"].unique()), name="step")
    pos = {}
    for team in ["blue", "red"]:
        units = alive[alive["team"] == team]
        pos[team] = np.stack(
            [units.pivot(index="step", columns="unit_id", values=c).reindex(steps).to_numpy() for c in "xy"],
            axis=-1,
        )  # step, unit, (x, y); NaN once a unit is dead
    gap = np.linalg.norm(
        pos["blue"][:, :, None, :] - pos["red"][:, None, :, :], axis=-1
    )  # step, Blue unit, Red unit
    within = gap <= REACH  # NaN compares False: dead units never count
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN steps once a side is gone
        front = np.nanmin(gap, axis=(1, 2))
        centre = {team: np.nanmean(p, axis=1) for team, p in pos.items()}
        axis = centre["red"] - centre["blue"]
        axis = axis / np.linalg.norm(axis, axis=1, keepdims=True)  # unit vector from Blue's centroid to Red's
        depth = {
            team: np.nanstd(np.einsum("sud,sd->su", p - centre[team][:, None, :], axis), axis=1)
            for team, p in pos.items()
        }
    return pd.DataFrame(
        {
            "front_distance": pd.Series(front, index=steps).ffill(),
            "reach_blue": within.any(axis=2).sum(axis=1).astype(float),
            "reach_red": within.any(axis=1).sum(axis=1).astype(float),
            "depth_blue": pd.Series(depth["blue"], index=steps).ffill(),
            "depth_red": pd.Series(depth["red"], index=steps).ffill(),
        },
        index=steps,
    )


def load_states(dataset_dir, episode_ids, n_chunks=10):
    # Team-level states on the common scale, built chunk by chunk to keep memory down
    parts = []
    for chunk in np.array_split(np.asarray(episode_ids), n_chunks):
        snaps = storage.load_snapshots(dataset_dir, chunk.tolist())
        by_episode = snaps.groupby("episode_idx")
        parts.append(
            by_episode.apply(level_a, include_groups=False)
            .join(by_episode.apply(engagement, include_groups=False))
            .join(by_episode.apply(geometry, include_groups=False))
        )
    states = pd.concat(parts)
    states[["health_blue", "health_red"]] /= REF_UNITS * MARINE_HP
    states[["alive_blue", "alive_red", "engaged_blue", "engaged_red", "reach_blue", "reach_red"]] /= REF_UNITS
    states["t"] = states.index.get_level_values("step") * dt
    return states


def fit_through_origin(feature, rate):
    # Least squares rate = c * feature, and the share of the rate's variance it explains
    c = np.dot(feature, rate) / np.dot(feature, feature)
    r2 = 1 - np.sum((rate - c * feature) ** 2) / np.sum((rate - rate.mean()) ** 2)
    return c, r2


states = load_states(DATASET_DIR, ids)

### Untrimmed trajectories and the split

Unlike the walkthrough, the battles are **not trimmed** to the first damage. The approach phase is
where distance changes most, and where the gate has to learn to "switch off" (no losses while
out of range). Battles shorter than one smoothing window are dropped. The split is by battle.

In [ ]:
trajectories = {
    i: to_trajectory(states.loc[i], ALL_COLUMNS, trim_to_engagement=False, downsample=1) for i in ids
}
trajectories = {
    i: tr for i, tr in trajectories.items() if len(tr) > SMOOTH_WINDOW
}  # the filter needs a full window

rng = np.random.default_rng(0)
shuffled = rng.permutation(sorted(trajectories))
n_test = len(shuffled) // 10
test_ids, train_ids = sorted(shuffled[:n_test].tolist()), sorted(shuffled[n_test:].tolist())
rows = pd.concat([trajectories[i] for i in train_ids])
print(f"{len(train_ids)} training and {len(test_ids)} test battles, {len(rows)} training rows")

## 2 · Measure s(d)

For every step where Red has units alive: the share of them with a target in range, averaged in
distance bins of 0.5. A marine's range is 5; with unit size and formation spread, the switch
from "in range" to "out of range" should sit a bit beyond that, measured between centroids.

In [ ]:
alive = rows[rows["alive_red"] > 0]
share = alive["engaged_red"] / alive["alive_red"]
bins = np.arange(0, 12.5, 0.5)
binned = share.groupby(pd.cut(alive["distance"], bins), observed=True).agg(["mean", "size"])
binned.round(2)

**Reading it** (5,000 battles):

- The share is about 0.9 at 4.5-5.5, 0.69 at 5.5-6, 0.34 at 6-6.5, 0.12 at 6.5-7, and about 0
  beyond 8.5: the switch sits around 6, a bit beyond a marine's per-unit range of 5.75 (5 plus two
  unit radii, centre to centre), because formations spread out.
- It also **falls below 4.5**: 0.75 at 4-4.5, 0.16 at 3.5-4, about 0 below 3.5 (about 5,000 rows,
  1.6%). So $s(d)$ is a band, not a step. Not checked yet why; a guess is the end of battles, when
  few units are left and the centroids say little about who is near whom. The sigmoid below
  ignores this.

## 3 · Fit a smooth step and test the closure

$s(d)$ is modelled as a sigmoid: 1 well inside distance $c$, 0 well outside, switching over a
width $w$:

$$s(d) = \frac{1}{1 + e^{(d - c)/w}}$$

`curve_fit` finds $c$ and $w$ per side. The R² says how much of the engaged count
$A \cdot s(d)$ explains; the rest is formation shape, which one centroid distance can't capture.

In [ ]:
def in_range_share(d, c, w):
    # Smooth step: 1 well inside distance c, 0 well outside, switching over a width w
    return 1 / (1 + np.exp((d - c) / w))


switch = {}
for side in ["red", "blue"]:
    a = rows[rows[f"alive_{side}"] > 0]
    (c, w), _ = curve_fit(
        in_range_share, a["distance"], a[f"engaged_{side}"] / a[f"alive_{side}"], p0=(6, 0.3)
    )
    switch[side] = (c, w)
    predicted = a[f"alive_{side}"] * in_range_share(a["distance"], c, w)
    actual = a[f"engaged_{side}"]
    r2 = 1 - np.sum((actual - predicted) ** 2) / np.sum((actual - actual.mean()) ** 2)
    print(f"{side}: switch at d = {c:.2f}, width {w:.2f}; engaged ≈ alive * s(d), R² = {r2:.2f}")

d = np.linspace(0, 12, 200)
plt.plot([b.mid for b in binned.index], binned["mean"], "o", label="data (Red)")
plt.plot(d, in_range_share(d, *switch["red"]), label="fitted s(d)")
plt.xlabel("distance between centroids")
plt.ylabel("share of alive units in range")
plt.legend();

**Reading it:** Red switches at 6.00 (width 0.32), Blue at 5.91 (width 0.44). The closure explains
R² 0.43 of Red's engaged count and 0.52 of Blue's: real, but far from all of it. The rest is
formation shape and target choice (only a unit's own target counts), plus the drop below 4.5
that the sigmoid misses.

## 4 · Does the closure explain losses?

The same regression as section 4 of the walkthrough (Blue's loss rate on one variable, through the
origin), with the closure added as a candidate. If `alive_red * s(d)` comes close to the measured
engaged count, distance and alive carry most of what engagement tells about losses. That's what
matters for a world model: future engagement can't be observed, but future distance can be modelled.

In [ ]:
X, DX = derivative_data(trajectories, train_ids, STATE, SMOOTH_WINDOW)

candidates = {
    "alive_red": rows["alive_red"],
    "engaged_red (measured)": rows["engaged_red"],
    "alive_red * s(d) (closure)": rows["alive_red"] * in_range_share(rows["distance"], *switch["red"]),
}
pd.DataFrame(
    {name: fit_through_origin(f.to_numpy(), DX[:, 0]) for name, f in candidates.items()},
    index=["coefficient", "R2"],
).T.round(3)

**Reading it:**

| feature | coefficient | R² |
|---|---|---|
| `alive_red` | -0.099 | 0.06 |
| `engaged_red` (measured) | -0.202 | 0.56 |
| `alive_red * s(d)` | -0.189 | 0.27 |

The closure keeps about **half** of what the measured engaged count explains, and its coefficient
is close (0.189 against 0.202, both near the full-firepower 0.219): the physics carries over, the
information doesn't fully. Alive alone explains almost nothing on untrimmed data, since the
approach phase has full alive counts and no losses.

## 5 · SINDy with a custom library

Instead of fixing the switch distance, give SINDy candidate features
$A \cdot s_c(d)$ for several switch distances $c$ and let STLSQ choose.

- **`CustomLibrary`** applies each two-argument function to every *pair* of the columns it is given.
- **`GeneralizedLibrary`** combines libraries; **`inputs_per_library`** restricts which columns each
  one sees. Without it you'd get features like "sigmoid of health". Given only `[3, 4]`, each
  function receives exactly `(alive_red, distance)`, in that order.
- **`u=`** passes alive and distance as inputs, so SINDy doesn't try to differentiate them. The
  columns the library sees are `[x, u]`: 0 `health_blue`, 1 `health_red`, 2 `alive_blue`,
  3 `alive_red`, 4 `distance`.
- A plain linear library (bias, health, alive) is included too, so SINDy can prefer it if the
  gates don't help.

In [ ]:
SWITCH_DISTANCES = [5.0, 5.5, 6.0, 6.5, 7.0]
WIDTH = 0.3


def make_gate(c):
    # A library function of two columns (alive, distance), and how it is named
    def function(alive, d):
        return alive / (1 + np.exp((d - c) / WIDTH))

    def name(alive, d):
        return f"{alive}*s{c}({d})"

    return function, name


gates = [make_gate(c) for c in SWITCH_DISTANCES]
gate_library = ps.CustomLibrary(library_functions=[f for f, _ in gates], function_names=[n for _, n in gates])

# Columns SINDy sees are [x, u]: 0 health_blue, 1 health_red, 2 alive_blue, 3 alive_red, 4 distance
library = ps.GeneralizedLibrary(
    [gate_library, gate_library, ps.PolynomialLibrary(degree=1)],
    inputs_per_library=[[3, 4], [2, 4], [0, 1, 2, 3]],  # Red's gates, Blue's gates, plain linear terms
)


def fit_with_inputs(library, threshold, inputs=INPUTS):
    model = ps.SINDy(
        optimizer=ps.STLSQ(threshold=threshold),
        feature_library=library,
        differentiation_method=ps.SmoothedFiniteDifference(
            smoother_kws={"window_length": SMOOTH_WINDOW, "polyorder": 2}
        ),
    )
    model.fit(
        [trajectories[i][STATE].to_numpy() for i in train_ids],
        t=dt,
        u=[trajectories[i][inputs].to_numpy() for i in train_ids],
        feature_names=STATE + inputs,
    )
    return model


fit_with_inputs(library, 0.05).get_feature_names()

## 6 · Threshold sweep

The physical expectation for Blue's losses: one term, `alive_red * s_c(distance)` with $c$ near the
fitted switch distance and a coefficient near a marine's damage rate, $6 / 0.61 / 45 \approx 0.22$
per second on this scale. pysindy's STLSQ uses **one threshold for both equations**.

In [ ]:
FULL_FIREPOWER = 6 / 0.61 / MARINE_HP

for threshold in [0.02, 0.05, 0.10, 0.15]:
    print(f"threshold {threshold}")
    fit_with_inputs(library, threshold).print(precision=3)

**Reading it:** no threshold gives a clean law.

- **Collinearity.** Gates half a unit apart are nearly the same function, so least squares trades
  them off: large terms of opposite sign that cancel (e.g. `+0.460 s5.0 - 0.437 s5.5`).
- **At 0.15** Blue's equation is `0.544 s5.0 - 0.559 s5.5`, i.e. about
  $-0.55\, A_R\,(s_{5.5}(d) - s_{5.0}(d))$: a **band** between distances 5 and 5.5, not a step. It
  matches the drop below 4.5 in section 2, and that's where most fighting rows lie. Read it as a
  hint, not a law.
- **Red's equation is eliminated entirely** at 0.15 (the warning): one threshold for both equations.
- On a 600-battle trial the same sweep gave Blue a single clean term at 0.15. Results this
  sensitive to the data are not a discovery.

Ways out: fewer, wider-spaced candidates (next cell), a threshold per equation, or ensembling.

### A one-gate library

One gate per side, at the switch distance from section 3 (rounded), plus the linear terms. With no
near-duplicate gates to trade off, a low threshold is enough.

In [ ]:
g = make_gate(6.0)
single_gate = ps.CustomLibrary(library_functions=[g[0]], function_names=[g[1]])
library_1 = ps.GeneralizedLibrary(
    [single_gate, single_gate, ps.PolynomialLibrary(degree=1)],
    inputs_per_library=[[3, 4], [2, 4], [0, 1, 2, 3]],
)
fit_with_inputs(library_1, 0.05).print(precision=3)

**Reading it:**

- `(health_blue)' = -0.187 alive_red*s6.0(distance)`: 85% of full firepower
- `(health_red)' = -0.203 alive_blue*s6.0(distance)`: 93%

One clean term per side and no linear terms left. That's the firepower law with engagement replaced
by alive × distance gate, found by STLSQ from a library that also offered health and alive.
Red's equation is now clean too, unlike the fit on Blue's measured engaged count. The switch
distance was chosen by hand (section 3), not by SINDy.

## 7 · Predicting ahead

`evaluate_horizons` can't pass inputs, so `horizon_errors_with_inputs` does the same scoring
(start points every second after 1 s of history, mean absolute error over both healths) and takes
the inputs over the horizon **from the data**. It steps forward with `model.predict` in Euler steps
of 0.125 s: close enough for health, and far faster than `simulate` with `u`.

Compared:

- **closure, one gate:** the one-gate model from section 6, with real future alive and distance
- **closure, five gates:** the threshold-0.15 model from the sweep, to see what its band term does
- **oracle engaged:** losses from the measured engaged counts, $\dot H_B = -k\,E_R$ (the walkthrough's
  firepower model, as a SINDy fit), with real future engagement
- **gated deg 1 and `last_slope`:** health-only, from `evaluate_horizons` on the same test battles

In [ ]:
def horizon_errors_with_inputs(model, ids, inputs, horizons=(0.5, 1.0, 2.0, 4.0), every=1.0, history=1.0):
    # Like evaluate_horizons, but the inputs over the horizon are taken from the data
    errors = {h: [] for h in horizons}
    for i in ids:
        traj = trajectories[i]
        x, u = traj[STATE].to_numpy(), traj[inputs].to_numpy()
        for k0 in range(round(history / dt), len(x) - round(horizons[0] / dt), round(every / dt)):
            xk = x[k0].copy()
            for n in range(1, round(horizons[-1] / dt) + 1):
                if k0 + n >= len(x):
                    break
                xk = np.clip(xk + dt * model.predict(xk[None, :], u=u[k0 + n - 1][None, :])[0], 0, None)
                if n * dt in errors:
                    errors[n * dt].append(np.mean(np.abs(xk - x[k0 + n])))
    return pd.Series({h: np.mean(e) for h, e in errors.items()})


engaged_library = ps.GeneralizedLibrary(
    [ps.PolynomialLibrary(degree=1, include_bias=False)], inputs_per_library=[[2, 3]]
)
gated = fit_gated(trajectories, train_ids, STATE, SMOOTH_WINDOW, degree=1, threshold=0.05)
baseline = evaluate_horizons(gated, test_ids, trajectories, STATE, every=1.0)

pd.DataFrame(
    {
        "closure, one gate": horizon_errors_with_inputs(fit_with_inputs(library_1, 0.05), test_ids, INPUTS),
        "closure, five gates (thr 0.15)": horizon_errors_with_inputs(
            fit_with_inputs(library, 0.15), test_ids, INPUTS
        ),
        "oracle: measured engaged": horizon_errors_with_inputs(
            fit_with_inputs(engaged_library, 0.02, ["engaged_blue", "engaged_red"]),
            test_ids,
            ["engaged_blue", "engaged_red"],
        ),
        "gated deg 1 (health only)": baseline["model"],
        "last_slope": baseline["last_slope"],
    }
).round(3)

**Reading it** (mean absolute error at 4 s):

| model | 4 s |
|---|---|
| oracle: measured engaged | 0.035 |
| closure, one gate | 0.061 |
| gated deg 1 (health only) | 0.065 |
| closure, five gates | 0.158 |
| `last_slope` | 0.101 |

- **The one-gate closure barely beats the health-only model**, even though it is given the *real*
  future alive and distance. Knowing the measured engagement halves the error; knowing alive and
  centroid distance gains almost nothing. Same story as section 4 (R² 0.27 against 0.56).
- **The five-gate model is worse than `last_slope`**: its Red equation was eliminated, so Red
  never loses health.
- So **centroid distance is too coarse** a stand-in for engagement: most of what engagement knows
  sits in formation geometry, not in the distance between team centres.

These errors are lower than the walkthrough's because the battles are untrimmed: the quiet approach
phase is easy to predict. Compare columns within this table only.

## 8 · Engagement as its own state, and why to wait

Engagement could also be a state that relaxes toward its geometric target with a time constant
$\tau$:

$$\dot E_R = \frac{A_R\, s(d) - E_R}{\tau}$$

The simplest estimate: the share of the gap $A_R s(d) - E_R$ that closes in one step.

In [ ]:
def target(tr):
    return make_gate(6.0)[0](tr["alive_red"].to_numpy(), tr["distance"].to_numpy())


E = np.concatenate([trajectories[i]["engaged_red"].to_numpy()[:-1] for i in train_ids])
E_next = np.concatenate([trajectories[i]["engaged_red"].to_numpy()[1:] for i in train_ids])
T = np.concatenate([target(trajectories[i])[:-1] for i in train_ids])

a = np.dot(T - E, E_next - E) / np.dot(T - E, T - E)  # share of the gap closed per step
print(f"closes {a:.2f} of the gap per step -> tau = {-dt / np.log(1 - a):.2f} s")

The same relaxation fitted through SINDy, with the derivative taken three ways:

In [ ]:
lag_library = ps.GeneralizedLibrary(
    [single_gate, ps.PolynomialLibrary(degree=1, include_bias=False)], inputs_per_library=[[1, 2], [0]]
)  # columns: 0 engaged_red, 1 alive_red, 2 distance
for name, differentiation in [
    ("smoothed, window 5", ps.SmoothedFiniteDifference(smoother_kws={"window_length": 5, "polyorder": 2})),
    ("finite differences", ps.FiniteDifference()),
]:
    lag = ps.SINDy(
        optimizer=ps.STLSQ(threshold=0.0), feature_library=lag_library, differentiation_method=differentiation
    )
    lag.fit(
        [trajectories[i][["engaged_red"]].to_numpy() for i in train_ids],
        t=dt,
        u=[trajectories[i][["alive_red", "distance"]].to_numpy() for i in train_ids],
        feature_names=["engaged_red", "alive_red", "distance"],
    )
    gain, decay = lag.coefficients()[0]
    print(f"{name}: tau = {-1 / decay:.2f} s (gain {-gain / decay:.2f})")

**Reading it:** the next-step estimate gives $\tau$ = 0.57 s; SINDy gives 6.8 s with a smoothed
derivative and 2.9 s with plain finite differences.

The spread means $\tau$ isn't identified from this jumpy count:

- the **next-step** estimate is biased fast: a one-step blip in the count reverts on its own, and
  that looks like quick relaxation
- **smoothing** is biased slow: it blurs out anything faster than its window

So engagement stays **algebraic** (sections 3-7). The relaxation version is only worth revisiting if
a planner needs steps under about 0.5 s.

## 9 · Better geometry: units within reach of an enemy

Sections 4 and 7 showed that centroid distance is too coarse. Two per-unit quantities, computed in
`geometry` from every unit's position:

- **`front_distance`:** the distance between the closest Blue and Red units, where the battle
  line is
- **`reach_*`:** the number of alive units with *any* enemy within `REACH` = 5.75, SMAClite's own
  in-range test. Pure geometry: unlike `engaged`, it ignores which unit a unit has chosen as target.

`reach` also splits each side into **two groups**, $A = \text{reach} + (A - \text{reach})$: units
that can shoot and units that can't. That is the smallest step towards a group-level state (Level
B, RQ2).

### How close is reach to engaged?

R² of `engaged ≈ reach` (as is, no fit), and the through-origin fit `engaged = c · reach`. A $c$
below 1 means some units with an enemy in reach have no target in range: they chose a target
further away, or none.

In [ ]:
pd.DataFrame(
    {
        side: {
            "R2, engaged = reach": 1
            - np.sum((rows[f"engaged_{side}"] - rows[f"reach_{side}"]) ** 2)
            / np.sum((rows[f"engaged_{side}"] - rows[f"engaged_{side}"].mean()) ** 2),
            **dict(
                zip(
                    ["c", "R2, engaged = c * reach"],
                    fit_through_origin(rows[f"reach_{side}"].to_numpy(), rows[f"engaged_{side}"].to_numpy()),
                    strict=False,
                )
            ),
        }
        for side in ["red", "blue"]
    }
).round(3)

**Reading it:** engaged ≈ 0.81 × reach for Red (R² 0.82) and 0.78 × reach for Blue (R² 0.71).
About one in five units with an enemy within reach is *not* engaged: its target is further away,
or it has none. For Blue that includes random actions and retreats.

### Which explains losses best?

Section 4's regression again, now with reach and the front distance closure. The real test of a
firepower variable is how much of the *opponent's loss rate* it explains, not how well it matches
`engaged`.

In [ ]:
for side in ["red", "blue"]:
    a = rows[rows[f"alive_{side}"] > 0]
    switch[f"{side}, front"], _ = curve_fit(
        in_range_share, a["front_distance"], a[f"engaged_{side}"] / a[f"alive_{side}"], p0=(5, 0.3)
    )

drivers = []
for side, rate, opponent in [("Blue", DX[:, 0], "red"), ("Red", DX[:, 1], "blue")]:
    candidates = {
        f"engaged_{opponent} (measured)": rows[f"engaged_{opponent}"],
        f"reach_{opponent}": rows[f"reach_{opponent}"],
        f"alive_{opponent} * s(centroid distance)": rows[f"alive_{opponent}"]
        * in_range_share(rows["distance"], *switch[opponent]),
        f"alive_{opponent} * s(front distance)": rows[f"alive_{opponent}"]
        * in_range_share(rows["front_distance"], *switch[f"{opponent}, front"]),
    }
    for name, f in candidates.items():
        c, r2 = fit_through_origin(f.to_numpy(), rate)
        drivers.append({"losses of": side, "explained by": name, "coefficient": c, "R2": r2})
pd.DataFrame(drivers).set_index(["losses of", "explained by"]).round(3)

**Reading it:**

| explained by | Blue's losses | Red's losses |
|---|---|---|
| measured engaged | 0.56 | 0.45 |
| **reach** | **0.73** | **0.72** |
| alive × s(centroid distance) | 0.27 | 0.53 |
| alive × s(front distance) | 0.60 | 0.62 |

(R² values; see the table for coefficients.)

- **Reach explains losses better than the measured engaged count**, for both sides, and the two
  sides now look alike. This is surprising, since `engaged` is closer to the mechanism. A likely
  reason (not checked): `engaged` flickers when targets die and units retarget, while damage keeps
  flowing on average; reach is a smoother count of the units that *can* shoot.
- The coefficients drop to 0.176 and 0.183 (80% and 84% of full firepower), because reach counts
  some units that aren't firing.
- The front distance is a better gate than the centroid distance, especially for Blue's losses.

### SINDy with reach in the library

A plain linear library on health, alive and reach (columns 0-5 of `[x, u]`): does STLSQ pick reach
on its own, out of everything it is offered?

In [ ]:
REACH_INPUTS = ["alive_blue", "alive_red", "reach_blue", "reach_red"]
# Columns SINDy sees: 0 health_blue, 1 health_red, 2 alive_blue, 3 alive_red, 4 reach_blue, 5 reach_red
reach_library = ps.GeneralizedLibrary(
    [ps.PolynomialLibrary(degree=1)], inputs_per_library=[[0, 1, 2, 3, 4, 5]]
)
for threshold in [0.02, 0.05]:
    print(f"threshold {threshold}")
    fit_with_inputs(reach_library, threshold, REACH_INPUTS).print(precision=3)

**Reading it:** at threshold 0.05 STLSQ keeps exactly one term per side:

- `(health_blue)' = -0.176 reach_red`
- `(health_red)' = -0.183 reach_blue`

Out of health, alive and reach, SINDy picks reach, and the law is symmetric between the sides. At
0.02 two small extra terms survive (-0.020 `health_blue`, -0.029 `alive_blue`). This is the
cleanest equation in the project so far, though reach is still read from the data.

### Can reach be predicted from distance?

Reach is still something a world model would have to predict. The same closure as section 3,
$\text{reach} \approx A \cdot s(d)$, now for the share of units within reach, against both distances.

In [ ]:
closure_fits = []
for side in ["red", "blue"]:
    a = rows[rows[f"alive_{side}"] > 0]
    for d_col in ["distance", "front_distance"]:
        (c, w), _ = curve_fit(in_range_share, a[d_col], a[f"reach_{side}"] / a[f"alive_{side}"], p0=(6, 0.3))
        switch[f"{side}, reach, {d_col}"] = (c, w)
        predicted = a[f"alive_{side}"] * in_range_share(a[d_col], c, w)
        actual = a[f"reach_{side}"]
        r2 = 1 - np.sum((actual - predicted) ** 2) / np.sum((actual - actual.mean()) ** 2)
        closure_fits.append({"side": side, "distance": d_col, "switch c": c, "width w": w, "R2": r2})
pd.DataFrame(closure_fits).set_index(["side", "distance"]).round(3)

In [ ]:
a = rows[rows["alive_red"] > 0]
reach_share = (a["reach_red"] / a["alive_red"]).groupby(pd.cut(a["distance"], bins), observed=True).mean()
plt.plot([b.mid for b in binned.index], binned["mean"], "o-", label="engaged / alive (section 2)")
plt.plot([b.mid for b in reach_share.index], reach_share, "s-", label="reach / alive")
plt.xlabel("distance between centroids")
plt.ylabel("share of Red's alive units")
plt.legend();

**Reading it:**

- Reach is easier to predict from distance than engaged was: R² 0.60 (Red) and 0.83 (Blue) from
  the centroid distance, against 0.43 and 0.52 for engaged in section 3. The front distance gives
  0.70 and 0.62, so neither distance wins for both sides.
- The plot: the reach share does **not** drop when the centroids are close; the engaged share
  does. So the band in section 2 is about target choice, not geometry: units with an enemy right
  next to them that are not shooting at a target in range. Probably Red's AI keeping a target
  further away, or late-battle stragglers; not checked.

### Predicting ahead with reach

Same scoring as section 7, all on the same test battles:

- **oracle reach:** losses from reach, with the real future reach
- **reach closure:** one gate per side at the switch distance fitted to reach (centroid distance),
  with real future alive and distance; once with the centroid distance, once with the front distance
- the section 7 models for comparison

In [ ]:
reach_gate = make_gate(round(switch["red, reach, distance"][0], 1))
reach_gate_library = ps.CustomLibrary(library_functions=[reach_gate[0]], function_names=[reach_gate[1]])
reach_closure_library = ps.GeneralizedLibrary(
    [reach_gate_library, reach_gate_library, ps.PolynomialLibrary(degree=1)],
    inputs_per_library=[[3, 4], [2, 4], [0, 1, 2, 3]],
)
reach_closure = fit_with_inputs(reach_closure_library, 0.05)
reach_closure.print(precision=3)

FRONT_INPUTS = ["alive_blue", "alive_red", "front_distance"]  # same column positions as INPUTS
front_gate = make_gate(round(switch["red, reach, front_distance"][0], 1))
front_gate_library = ps.CustomLibrary(library_functions=[front_gate[0]], function_names=[front_gate[1]])
front_closure = fit_with_inputs(
    ps.GeneralizedLibrary(
        [front_gate_library, front_gate_library, ps.PolynomialLibrary(degree=1)],
        inputs_per_library=[[3, 4], [2, 4], [0, 1, 2, 3]],
    ),
    0.05,
    FRONT_INPUTS,
)
front_closure.print(precision=3)

oracle_reach = fit_with_inputs(reach_library, 0.05, REACH_INPUTS)
pd.DataFrame(
    {
        "oracle: reach": horizon_errors_with_inputs(oracle_reach, test_ids, REACH_INPUTS),
        "oracle: measured engaged": horizon_errors_with_inputs(
            fit_with_inputs(engaged_library, 0.02, ["engaged_blue", "engaged_red"]),
            test_ids,
            ["engaged_blue", "engaged_red"],
        ),
        "reach closure: alive * s(centroid d)": horizon_errors_with_inputs(reach_closure, test_ids, INPUTS),
        "reach closure: alive * s(front d)": horizon_errors_with_inputs(
            front_closure, test_ids, FRONT_INPUTS
        ),
        "engaged closure, one gate": horizon_errors_with_inputs(
            fit_with_inputs(library_1, 0.05), test_ids, INPUTS
        ),
        "gated deg 1 (health only)": baseline["model"],
        "last_slope": baseline["last_slope"],
    }
).round(3)

**Reading it** (mean absolute error at 4 s; the fitted closures print above the table):

| model | 4 s |
|---|---|
| oracle: reach | **0.031** |
| oracle: measured engaged | 0.035 |
| reach closure, centroid distance | 0.055 |
| reach closure, front distance | 0.059 |
| engaged closure, one gate | 0.061 |
| gated deg 1 (health only) | 0.065 |
| `last_slope` | 0.101 |

- **Knowing reach gives the best predictions in this notebook**, better than knowing measured
  engagement. Reach is the variable worth modelling.
- **Predicting reach from one distance still recovers only part of that gain.** The centroid closure
  closes about 30% of the gap between the health-only model and oracle reach
  ((0.065 - 0.055) / (0.065 - 0.031)), and that is with the real future alive and distance. The
  front distance closure fits better per step (R² above) but predicts slightly worse.

## 10 · Reach as a state: an equation for it

Section 9 found the variable to model: health follows the opponent's reach. A world model must
then predict reach itself. Where does an equation for it come from?

If reach follows the closure $R \approx A \cdot s(d)$ at every moment, differentiating it (chain
rule) gives

$$\dot R \approx \underbrace{A\, s'(d)\, \dot d}_{\text{movement}} + \underbrace{\dot A\, s(d)}_{\text{deaths}}$$

- **movement:** units cross the reach boundary as the teams close in ($\dot d < 0$, and
  $s'(d) < 0$, so reach grows) or separate
- **deaths:** units in reach die at the rate alive falls, times the share in reach

Both should come out with coefficient 1. Unlike the closure, an equation keeps each battle's
own **offset**: if a battle starts with fewer units in reach than $A \cdot s(d)$ says (a wide
formation, say), the equation carries that on.

First: how jumpy is reach?

In [ ]:
changes = np.concatenate([np.diff(trajectories[i]["reach_red"].to_numpy()) for i in train_ids])
share_changed = np.mean(changes != 0)
mean_jump = np.abs(changes[changes != 0]).mean() * REF_UNITS
print(f"reach_red changes in {share_changed:.0%} of steps, by {mean_jump:.1f} units on average when it does")

**Reading it:** Red's reach changes in 34% of steps, by 1.4 units on average when it does. It is
a count that moves in jumps, so its derivative is noisy, though less so than engaged's (section 8).

### The candidate features

Computed per battle, with the same Savitzky-Golay filter as the health derivatives for the rates
$\dot d$ and $\dot A$, and $s$ the reach share fitted in section 9 (centroid distance), with
$s'(d) = -s(1 - s)/w$. `target_*` is the closure $A \cdot s(d)$ itself.

In [ ]:
rate_filter = ps.SmoothedFiniteDifference(smoother_kws={"window_length": SMOOTH_WINDOW, "polyorder": 2})


def reach_features(traj):
    # Movement and death terms of the differentiated closure, per side
    t, d = traj["t"].to_numpy(), traj["distance"].to_numpy()

    def rate(col):
        return np.asarray(rate_filter(traj[[col]].to_numpy(), t=t))[:, 0]

    d_rate = rate("distance")
    features = {}
    for side in ["blue", "red"]:
        c, w = switch[f"{side}, reach, distance"]
        s_d, alive = in_range_share(d, c, w), traj[f"alive_{side}"].to_numpy()
        features[f"move_{side}"] = alive * (-s_d * (1 - s_d) / w) * d_rate
        features[f"deaths_{side}"] = rate(f"alive_{side}") * s_d
        features[f"target_{side}"] = alive * s_d
    return traj.assign(**features)


trajectories = {i: reach_features(traj) for i, traj in trajectories.items()}

### SINDy, offered everything

One fit per side (`x = [reach_side]`), so each equation only sees its own side's features plus
the opponent's reach. Offered: own reach, movement, deaths, the closure target, alive, and the
opponent's reach (which kills units in reach).

In [ ]:
def fit_reach(side, inputs, threshold):
    model = ps.SINDy(
        optimizer=ps.STLSQ(threshold=threshold),
        feature_library=ps.PolynomialLibrary(degree=1, include_bias=False),
        differentiation_method=ps.SmoothedFiniteDifference(
            smoother_kws={"window_length": SMOOTH_WINDOW, "polyorder": 2}
        ),
    )
    model.fit(
        [trajectories[i][[f"reach_{side}"]].to_numpy() for i in train_ids],
        t=dt,
        u=[trajectories[i][inputs].to_numpy() for i in train_ids],
        feature_names=[f"reach_{side}"] + inputs,
    )
    return model


def all_reach_inputs(side):
    opponent = "red" if side == "blue" else "blue"
    return [f"move_{side}", f"deaths_{side}", f"target_{side}", f"alive_{side}", f"reach_{opponent}"]


for threshold in [0.05, 0.2]:
    print(f"threshold {threshold}")
    for side in ["blue", "red"]:
        fit_reach(side, all_reach_inputs(side), threshold).print(precision=3)

**Reading it:**

- **Deaths come out near 1** (0.88-1.01) and **movement at 0.5-0.7**, in every fit.
- **Blue's equation has a positive term in its own reach (0.39-0.55) together with a negative one
  on the target (-0.63 to -0.65)**: roughly $\dot R \propto +(R - A s(d))$, which pushes reach
  *away* from the closure. Integrated, that grows without limit: on a 600-battle trial it predicted
  worse than holding reach constant at 4 s. It is the same identification problem as section 8: a
  persistent offset between reach and the closure looks like growth to a regression.
- At 0.2 Red's equation drops to the two chain-rule terms on its own; Blue's doesn't.

### SINDy, offered only the chain-rule terms

The same fit with only movement and deaths: the differentiated closure, with SINDy finding the two
coefficients. The library still offers reach itself (`PolynomialLibrary` takes every column of
`[x, u]`), so STLSQ may keep a term in it.

In [ ]:
chain_inputs = {side: [f"move_{side}", f"deaths_{side}"] for side in ["blue", "red"]}
reach_models = {side: fit_reach(side, chain_inputs[side], 0.05) for side in ["blue", "red"]}
for model in reach_models.values():
    model.print(precision=3)

**Reading it:**

- `(reach_blue)' = -0.065 reach_blue + 0.790 move_blue + 0.875 deaths_blue`
- `(reach_red)' = 0.607 move_red + 1.012 deaths_red`

**Deaths at 0.88-1.01**, as the chain rule says. **Movement at 0.6-0.8**, below 1: probably the
noisy rate of distance attenuating its coefficient, or units crossing the boundary at different
moments than the centroids suggest; not checked. The library still offers reach itself, and for
Blue STLSQ keeps a small -0.065: a slow decay, which keeps the equation stable.

### Health and reach together, a few seconds ahead

The coupled model, stepped forward from the real state at each start point:

$$\dot H_B = -k_B\, R_R, \qquad \dot H_R = -k_R\, R_B, \qquad \dot R = \text{the chain-rule equation}$$

with $k$ the through-origin fits from section 9. Movement and deaths come from the data (they
need $\dot d$ and $\dot A$, which nothing predicts yet). Four ways to get reach over the horizon,
on the same start points:

- **model:** the chain-rule equation, integrated
- **data:** the real reach (the oracle from section 9)
- **constant:** reach held at its start value
- **closure:** $A \cdot s(d)$ at each step, from the real future alive and distance

Scored on health (comparable with sections 7 and 9) and on reach itself.

In [ ]:
k_reach = {
    "blue": -fit_through_origin(rows["reach_red"].to_numpy(), DX[:, 0])[
        0
    ],  # Blue's losses per Red unit in reach
    "red": -fit_through_origin(rows["reach_blue"].to_numpy(), DX[:, 1])[0],
}


def coupled_errors(ids, reach_from, horizons=(0.5, 1.0, 2.0, 4.0), every=1.0, history=1.0):
    # Mean absolute error of health and of reach, with reach over the horizon taken from `reach_from`
    health_errors, reach_errors = {h: [] for h in horizons}, {h: [] for h in horizons}
    for i in ids:
        traj = trajectories[i]
        H, R = traj[STATE].to_numpy(), traj[["reach_blue", "reach_red"]].to_numpy()
        closure = traj[["target_blue", "target_red"]].to_numpy()
        u = {side: traj[chain_inputs[side]].to_numpy() for side in ["blue", "red"]}
        for k0 in range(round(history / dt), len(H) - round(horizons[0] / dt), round(every / dt)):
            h, r = H[k0].copy(), R[k0].copy()
            for n in range(1, round(horizons[-1] / dt) + 1):
                if k0 + n >= len(H):
                    break
                r_now = {
                    "model": r,
                    "data": R[k0 + n - 1],
                    "constant": R[k0],
                    "closure": closure[k0 + n - 1],
                }[reach_from]
                h = np.clip(
                    h + dt * np.array([-k_reach["blue"] * r_now[1], -k_reach["red"] * r_now[0]]), 0, None
                )
                if reach_from == "model":
                    dr = [
                        reach_models[side].predict(r[[j]][None, :], u=u[side][k0 + n - 1][None, :])[0, 0]
                        for j, side in enumerate(["blue", "red"])
                    ]
                    r = np.clip(r + dt * np.array(dr), 0, None)
                r_end = {"model": r, "data": R[k0 + n], "constant": R[k0], "closure": closure[k0 + n]}[
                    reach_from
                ]
                if n * dt in health_errors:
                    health_errors[n * dt].append(np.mean(np.abs(h - H[k0 + n])))
                    reach_errors[n * dt].append(np.mean(np.abs(r_end - R[k0 + n])))
    return pd.Series({x: np.mean(e) for x, e in health_errors.items()}), pd.Series(
        {x: np.mean(e) for x, e in reach_errors.items()}
    )


coupled = {name: coupled_errors(test_ids, name) for name in ["model", "data", "constant", "closure"]}
pd.concat(
    {
        "health error": pd.DataFrame({name: e[0] for name, e in coupled.items()}),
        "reach error": pd.DataFrame({name: e[1] for name, e in coupled.items()}),
    },
    axis=1,
).round(3)

**Reading it** (health error, then reach error):

| reach from | health 1 s | health 4 s | reach 1 s | reach 4 s |
|---|---|---|---|---|
| data (oracle) | 0.015 | **0.031** | 0 | 0 |
| **model** (chain-rule equation) | **0.018** | 0.053 | **0.079** | 0.089 |
| closure $A \cdot s(d)$ | 0.021 | 0.053 | 0.081 | **0.078** |
| constant | 0.020 | 0.087 | 0.118 | 0.210 |

- **The reach equation is stable and useful:** far better than holding reach constant, and the
  best non-oracle health prediction up to 2 s (0.029 against 0.032 for the closure at 2 s).
- **At 4 s it ties the closure on health (0.053) and loses on reach (0.089 against 0.078).** The
  offset it carries helps for a second or two, then stops being true: formations change.
- **The gap to the oracle stays.** At 4 s, both reach the same 0.053 against 0.031 with the real
  reach. Alive and centroid distance hold only part of what reach knows, whether used as a
  closure or as an equation.

## 11 · Formation spread

Section 10 ended at a ceiling: alive and centroid distance don't say enough about reach. The
missing piece is the shape of the formations. Picture each formation as units spread along the
line between the two teams:

- **The enemy's depth** moves its front line towards you: its closest units are ahead of its
  centroid, so you get within reach at a larger centroid distance. It should **shift** the switch
  outward.
- **Your own depth** means your units cross the boundary at different moments: the front rows
  first, the back rows later. It should **widen** the switch.

So the share in reach becomes

$$s(d) = \frac{1}{1 + \exp\!\Big(\frac{d - c - b_{own}\,\sigma_{own} - b_{opp}\,\sigma_{opp}}{w_0 + a\,\sigma_{own}}\Big)}$$

with $\sigma$ = `depth_*`. For a formation of zero depth facing one of zero depth, the switch
should sit at exactly `REACH` = 5.75: $c \approx 5.75$ is a check on the physics.

In [ ]:
rows = pd.concat(
    [trajectories[i] for i in train_ids]
)  # again: the trajectories now hold section 10's columns too
rows[["depth_blue", "depth_red"]].describe().round(2)

**Reading it:** a typical depth is about 0.35 map units (standard deviation along the line), about
half a marine's diameter: formations two or three rows deep. Blue's has a long tail (up to 7.3):
retreating units and stragglers stretch Blue's formation. Red's stays below 2.

In [ ]:
def in_reach_share(X, c, b_own, b_opp, w0, a):
    # Share of alive units within reach, from centroid distance and both formations' depth
    d, depth_own, depth_opp = X
    return 1 / (
        1 + np.exp((d - c - b_own * depth_own - b_opp * depth_opp) / np.maximum(w0 + a * depth_own, 0.01))
    )


def count_r2(actual, alive, share):
    # R² of the predicted count alive * share against the actual count
    return 1 - np.sum((actual - alive * share) ** 2) / np.sum((actual - actual.mean()) ** 2)


spread_fits, spread_params = [], {}
for side, opponent in [("red", "blue"), ("blue", "red")]:
    a = rows[rows[f"alive_{side}"] > 0].dropna(subset=["distance", f"depth_{side}", f"depth_{opponent}"])
    X = (a["distance"].to_numpy(), a[f"depth_{side}"].to_numpy(), a[f"depth_{opponent}"].to_numpy())
    params, _ = curve_fit(
        in_reach_share, X, a[f"reach_{side}"] / a[f"alive_{side}"], p0=(6, 0, 0, 0.3, 0), maxfev=20000
    )
    spread_params[side] = params
    actual, alive = a[f"reach_{side}"], a[f"alive_{side}"]
    spread_fits.append(
        {
            "side": side,
            **dict(zip(["c", "b_own", "b_opp", "w0", "a"], params, strict=False)),
            "R2 with spread": count_r2(actual, alive, in_reach_share(X, *params)),
            "R2 without (section 9)": count_r2(
                actual, alive, in_range_share(a["distance"], *switch[f"{side}, reach, distance"])
            ),
        }
    )
pd.DataFrame(spread_fits).set_index("side").round(3)

**Reading it:**

| | switch c | enemy depth b_opp | own depth widens by a | R² with spread | R² without |
|---|---|---|---|---|---|
| Red | **5.76** | 1.44 | 0.75 | **0.74** | 0.60 |
| Blue | **5.74** | 1.43 | 0.61 | **0.92** | 0.83 |

- **The switch lands on the per-unit reach.** $c$ is a free parameter, and it comes out at 5.76 and
  5.74 against `REACH` = 5.75. Once spread is accounted for, the centroid law is the unit law.
- **Enemy depth shifts the switch outward by about 1.4 per unit of depth**, the same for both sides:
  the enemy's front units sit about 1.4 standard deviations ahead of its centroid.
- **Own depth sets the width** ($w_0 \approx 0$): a formation with no depth would switch sharply.
  $b_{own}$ is small.
- Every parameter has a physical reading, and the fit improves for both sides.

### Predicting ahead with spread

The closure from section 10, now with spread: reach over the horizon is $A \cdot s(d, \sigma)$
from the real future alive, distance and depths. Same start points and the same $k$ as section 10,
so the columns compare directly with its table.

In [ ]:
for i, traj in trajectories.items():
    X = {
        side: (
            traj["distance"].to_numpy(),
            traj[f"depth_{side}"].to_numpy(),
            traj[f"depth_{opponent}"].to_numpy(),
        )
        for side, opponent in [("blue", "red"), ("red", "blue")]
    }
    trajectories[i] = traj.assign(
        **{
            f"target_{side}": traj[f"alive_{side}"].to_numpy() * in_reach_share(X[side], *spread_params[side])
            for side in ["blue", "red"]
        }
    )  # coupled_errors' "closure" now uses the spread closure

spread_closure = coupled_errors(test_ids, "closure")
pd.concat(
    {
        "health error": pd.DataFrame(
            {
                "data (oracle)": coupled["data"][0],
                "closure with spread": spread_closure[0],
                "closure, distance only": coupled["closure"][0],
                "reach equation": coupled["model"][0],
            }
        ),
        "reach error": pd.DataFrame(
            {
                "closure with spread": spread_closure[1],
                "closure, distance only": coupled["closure"][1],
                "reach equation": coupled["model"][1],
            }
        ),
    },
    axis=1,
).round(3)

**Reading it:**

| | health 1 s | health 4 s | reach 1 s | reach 4 s |
|---|---|---|---|---|
| oracle: real reach | 0.015 | **0.031** | 0 | 0 |
| **closure with spread** | **0.018** | **0.044** | **0.054** | **0.054** |
| closure, distance only | 0.021 | 0.053 | 0.081 | 0.078 |
| reach equation (section 10) | 0.018 | 0.053 | 0.079 | 0.089 |

- **Spread is the missing piece.** At 4 s the health error drops from 0.053 to 0.044, and the reach
  error by about a third at every horizon.
- Against the health-only gated model's 0.065 (section 7; scored on the same battles but with its
  own integrator, so roughly comparable), the spread closure recovers about **60%** of what knowing
  the real reach is worth, against about 35% for distance alone.
- **Still an oracle on the inputs:** the real future alive, distance and depths. Depth now needs
  an equation of its own too.

## 12 · A fair test: no future data

Every prediction so far read the **real future** alive, distance and depth. Here nothing after the
start point is read. Reach over the horizon comes from the spread closure (section 11), with its
inputs guessed from what is known at the start:

- **geometry** (distance and both depths): **hold** at the start value, or **trend**: the start
  value plus the slope over the last second (the same 1 s of history every start point has)
- **alive:** **hold** at the start value, or **from health**: the start value scaled by predicted
  health over start health, so units "die" as the model's own health falls
- **+ offset:** also carry the start point's gap between real reach and the closure

Same start points, the same $k$ and the same Euler steps as sections 10-11. The first row reads the
real future and should reproduce section 11's 0.044.

In [ ]:
GEOM = ["distance", "depth_blue", "depth_red"]
ALIVE = ["alive_blue", "alive_red"]


def reach_from(geom, alive):
    # The spread closure (section 11): units in reach from distance, both depths and alive
    d, depth_blue, depth_red = geom
    with np.errstate(over="ignore"):  # exp overflows far out of reach, where the share is 0 anyway
        return np.array(
            [
                alive[0] * in_reach_share((d, depth_blue, depth_red), *spread_params["blue"]),
                alive[1] * in_reach_share((d, depth_red, depth_blue), *spread_params["red"]),
            ]
        )


def no_future_errors(
    ids, geometry, alive, offset=False, horizons=(0.5, 1.0, 2.0, 4.0), every=1.0, history=1.0
):
    # Health error with reach from the spread closure, and its inputs over the horizon from:
    #   geometry: "data" (real future), "hold" (start value) or "trend" (start value plus the slope
    #             over the last `history` seconds)
    #   alive:    "data", "hold", or "health" (start value scaled by predicted health / start health)
    #   offset:   add the start point's gap between real reach and the closure, held over the horizon
    errors = {h: [] for h in horizons}
    m = round(history / dt)
    for i in ids:
        traj = trajectories[i]
        H, A = traj[STATE].to_numpy(), traj[ALIVE].to_numpy()
        G = traj[GEOM].ffill().bfill().to_numpy()
        R = traj[["reach_blue", "reach_red"]].to_numpy()
        for k0 in range(m, len(H) - round(horizons[0] / dt), round(every / dt)):
            slope = (G[k0] - G[k0 - m]) / (m * dt)
            gap = R[k0] - reach_from(G[k0], A[k0]) if offset else 0
            h = H[k0].copy()
            for n in range(1, round(horizons[-1] / dt) + 1):
                if k0 + n >= len(H):
                    break
                k = k0 + n - 1  # inputs at the start of the step, as in coupled_errors
                g = {"data": G[k], "hold": G[k0], "trend": np.maximum(G[k0] + slope * (k - k0) * dt, 0)}[
                    geometry
                ]
                a = {
                    "data": A[k],
                    "hold": A[k0],
                    "health": A[k0] * np.divide(h, H[k0], out=np.zeros(2), where=H[k0] > 0),
                }[alive]
                r = np.clip(reach_from(g, a) + gap, 0, None)
                h = np.clip(h + dt * np.array([-k_reach["blue"] * r[1], -k_reach["red"] * r[0]]), 0, None)
                if n * dt in errors:
                    errors[n * dt].append(np.mean(np.abs(h - H[k0 + n])))
    return pd.Series({x: np.mean(e) for x, e in errors.items()})


fair = {
    "real geometry and alive (section 11)": no_future_errors(test_ids, "data", "data"),
    "hold geometry, hold alive": no_future_errors(test_ids, "hold", "hold"),
    "hold geometry, alive from health": no_future_errors(test_ids, "hold", "health"),
    "trend geometry, hold alive": no_future_errors(test_ids, "trend", "hold"),
    "trend geometry, alive from health": no_future_errors(test_ids, "trend", "health"),
    "trend geometry, alive from health, + offset": no_future_errors(test_ids, "trend", "health", offset=True),
    "hold real reach (section 10)": coupled["constant"][0],
    "gated deg 1 (health only)": baseline["model"],
    "last_slope": baseline["last_slope"],
}
pd.DataFrame(fair).T.round(3)

**Reading it** (mean absolute health error):

| reach from | 1 s | 2 s | 4 s |
|---|---|---|---|
| real geometry and alive (oracle inputs) | 0.018 | 0.026 | 0.044 |
| **trend geometry, alive from health** | **0.019** | **0.033** | **0.057** |
| … + offset | 0.019 | 0.033 | 0.059 |
| hold geometry, alive from health | 0.022 | 0.040 | 0.084 |
| hold geometry, hold alive | 0.022 | 0.042 | 0.092 |
| trend geometry, hold alive | 0.021 | 0.048 | 0.117 |
| hold real reach (section 10) | 0.020 | 0.040 | 0.087 |
| gated deg 1 (health only) | 0.024 | 0.039 | 0.065 |
| `last_slope` | 0.027 | 0.052 | 0.101 |

- **The reach model now beats the health-only model with no future data**: 0.057 against 0.065 at
  4 s, 0.033 against 0.039 at 2 s. That is the first fair win for the physical model. It recovers
  about a quarter of the gap to the oracle reach ((0.065 - 0.057) / (0.065 - 0.031)).
- **Both pieces are needed.** Holding the geometry is worse than the health-only model, since the
  teams keep closing in. Extending the trend *without* deaths is worst of all (0.117): reach
  keeps growing while units in reach should be dying. With alive tied to the model's own health,
  deaths cap the growth.
- **The offset doesn't help**: the closure's error is not a steady per-battle bias.
- The health-only model has its own integrator (section 7), so it compares roughly, not exactly.
- Still crude: a straight-line trend in distance overshoots once the teams meet, and alive is a
  smooth share, not a count. Equations for distance and depth should do better than a trend.

## 13 · Closing the loop: an equation for distance

Section 12 still guessed the geometry. Here distance gets an equation of its own, and the model
runs from the start state alone: health, alive, distance and depth at the start point, plus Blue's
policy setting.

Neither side has a movement intent in exp05: Red is the game AI, attack-moving, and Blue walks
towards its target. So the expectation is simple: the teams close at marching speed until they
are in contact, then stop.

First, the closing speed against distance, from the same smoothed derivative as section 10, on
training battles while both sides have units alive. Blue's setting per battle comes from the
`controls` table; with `aggression` < 1 a share of Blue's actions is random, which should slow the
approach (`timid` = 1 − aggression).

In [ ]:
controls = storage.load_controls(DATASET_DIR, ids).groupby("episode_idx").first()  # one setting per battle


def closing_rates(i):
    # Per step: distance, depths, the rate of distance, and Blue's setting
    traj = trajectories[i]
    geom = traj[GEOM].ffill().bfill()
    return pd.DataFrame(
        {
            **{col: geom[col].to_numpy() for col in GEOM},
            "rate": np.asarray(rate_filter(geom[["distance"]].to_numpy(), t=traj["t"].to_numpy()))[:, 0],
            "timid": 1 - controls.loc[i, "aggression"],
            "rule": controls.loc[i, "rule"],
            "both_alive": (traj["alive_blue"].to_numpy() > 0) & (traj["alive_red"].to_numpy() > 0),
        }
    )


moves = pd.concat([closing_rates(i) for i in train_ids]).query("both_alive").dropna()
far = moves[moves["distance"] > 10]
display(moves.groupby(pd.cut(moves["distance"], np.arange(3, 16, 1)), observed=True)["rate"].mean().round(2))
display(
    far.groupby(pd.cut(far["timid"], [0, 0.1, 0.2, 0.3], include_lowest=True), observed=True)["rate"]
    .agg(["mean", "size"])
    .round(2)
)
far.groupby("rule")["rate"].mean().round(2)

**Reading it:**

- Far apart, the teams close at about **5.5 units per second** (two marines walking towards each
  other). The speed falls between distance 9 and 6 and is near zero in contact (below 6).
- **Aggression sets the speed:** 5.9 for the most aggressive Blue, 5.2 for the most timid. The
  random actions cost Blue walking time. The target rule makes no difference.

### The law, and does SINDy pick it?

The same smooth step as sections 3 and 11, now for movement: closing speed $v$ far away, switching
off around a distance $c$, shifted by the formations' depth:

$$\dot d = -\,\frac{v}{1 + e^{-(d - c - b(\sigma_B + \sigma_R))/w}} = -v\, g(d, \sigma)$$

`curve_fit` gives the shape ($c$, $b$, $w$); then SINDy is offered the gate $g$, `timid` × $g$
and everything else at hand (distance, depths, alive, reach), and chooses.

In [ ]:
def closing_gate(X, v, c, b, w):
    # Share of full closing speed: 1 far apart, 0 in contact
    d, depth_blue, depth_red = X
    with np.errstate(over="ignore"):
        return v / (1 + np.exp(-(d - c - b * (depth_blue + depth_red)) / w))


X_moves = tuple(moves[col].to_numpy() for col in GEOM)
(v_close, *gate_shape), _ = curve_fit(
    lambda X, v, c, b, w: -closing_gate(X, v, c, b, w), X_moves, moves["rate"], p0=(5.4, 7.5, 0, 0.5)
)
r2 = 1 - np.sum((moves["rate"] + closing_gate(X_moves, v_close, *gate_shape)) ** 2) / np.sum(
    (moves["rate"] - moves["rate"].mean()) ** 2
)
print(
    f"v = {v_close:.2f}, c = {gate_shape[0]:.2f}, b = {gate_shape[1]:.2f}, "
    f"w = {gate_shape[2]:.2f}; R² = {r2:.2f}"
)


def distance_inputs(i):
    # Columns SINDy sees after distance: depths, gate, timid * gate, alive and reach of both sides
    traj = trajectories[i]
    geom = traj[GEOM].ffill().bfill().fillna(0)
    g = closing_gate(tuple(geom[col].to_numpy() for col in GEOM), 1.0, *gate_shape)
    return np.column_stack(
        [geom["depth_blue"], geom["depth_red"], g, (1 - controls.loc[i, "aggression"]) * g]
        + [traj[col] for col in ALIVE + ["reach_blue", "reach_red"]]
    )


DISTANCE_FEATURES = (
    ["distance", "depth_blue", "depth_red", "gate", "timid*gate"] + ALIVE + ["reach_blue", "reach_red"]
)
for threshold in [0.1, 0.5]:
    print(f"threshold {threshold}")
    distance_model = ps.SINDy(
        optimizer=ps.STLSQ(threshold=threshold),
        feature_library=ps.PolynomialLibrary(degree=1),
        differentiation_method=ps.SmoothedFiniteDifference(
            smoother_kws={"window_length": SMOOTH_WINDOW, "polyorder": 2}
        ),
    )
    distance_model.fit(
        [trajectories[i][["distance"]].ffill().bfill().to_numpy() for i in train_ids],
        t=dt,
        u=[distance_inputs(i) for i in train_ids],
        feature_names=DISTANCE_FEATURES,
    )
    distance_model.print(precision=3)

**Reading it:**

- The fit: $v$ = 5.44, switch at $c$ = 8.34, width 0.49, R² 0.82. The switch sits well beyond the
  reach of 5.75: the centroids are about 8.3 apart when the front rows meet, and the back rows
  catch up slowly.
- **At threshold 0.5 SINDy keeps exactly the two physical terms:**
  `(distance)' = -5.78 gate + 2.18 timid*gate`. Closing speed is $5.78 - 2.18\,(1 - \text{aggression})$
  times the gate. Blue's aggression enters as a **control input**, with a sign and size that make
  sense: at aggression 0.7 Blue closes about 11% slower.
- At 0.1 small terms in depth, alive and reach survive (Blue's reach slows the approach, Red's
  speeds it up). Not pursued.
- $b$ is negative (−0.57): deeper formations close to a slightly *smaller* centroid distance. Not
  checked why.

### Which future input matters?

Before using the equation: section 12 guessed all three inputs at once. Here each input is in turn
real, held, or extended along its trend, with the others guessed. `rollout` steps the whole model
forward from a start point; `distance="model"` integrates the equation above (the threshold-0.5 fit).

In [ ]:
def rollout(i, k0, n_steps, distance="model", depth="hold", alive="health", history=1.0, tau=0.5):
    # Health and distance over n_steps from step k0. Each input comes from "data" (real future),
    # "hold" (start value) or "trend" (start value plus the slope over the last `history` s);
    # alive can also be "health" (scaled by predicted health), and distance
    #   "model": the distance equation, d' = v*(d)
    #   "model + speed": closing speed v as a state, starting at the observed slope and relaxing
    #                    towards the equation's v*(d) with time constant tau: d' = v, v' = (v* - v) / tau
    traj = trajectories[i]
    H, A = traj[STATE].to_numpy(), traj[ALIVE].to_numpy()
    G = traj[GEOM].ffill().bfill().to_numpy()
    m = round(history / dt)
    slope = (G[k0] - G[k0 - m]) / (m * dt)
    inputs = np.zeros((1, len(DISTANCE_FEATURES) - 1))
    h, d, v = H[k0].copy(), G[k0, 0], slope[0]
    from_model = distance.startswith("model")
    healths, distances = [h.copy()], [d]
    for n in range(1, n_steps + 1):
        k = k0 + n - 1  # inputs at the start of the step
        guess = {"data": G[k], "hold": G[k0], "trend": np.maximum(G[k0] + slope * (k - k0) * dt, 0)}
        d_now = d if from_model else guess[distance][0]
        g = np.array([d_now, *guess[depth][1:]])
        a = {
            "data": A[k],
            "hold": A[k0],
            "health": A[k0] * np.divide(h, H[k0], out=np.zeros(2), where=H[k0] > 0),
        }[alive]
        r = np.clip(reach_from(g, a), 0, None)
        h = np.clip(h + dt * np.array([-k_reach["blue"] * r[1], -k_reach["red"] * r[0]]), 0, None)
        if from_model:
            gate = closing_gate((d, *g[1:]), 1.0, *gate_shape)
            inputs[0, 2:4] = gate, (1 - controls.loc[i, "aggression"]) * gate  # only these two terms are kept
            v_star = distance_model.predict(np.array([[d]]), u=inputs)[0, 0]
            if distance == "model":
                d = max(d + dt * v_star, 0)
            else:
                d, v = max(d + dt * v, 0), v + dt * (v_star - v) / tau
            distances.append(d)
        else:  # the guess one step on, so the distance error is scored at the same time as health
            distances.append(
                {"data": G[k0 + n], "hold": G[k0], "trend": np.maximum(G[k0] + slope * n * dt, 0)}[distance][
                    0
                ]
            )
        healths.append(h.copy())
    return np.array(healths), np.array(distances)


def rollout_errors(ids, horizons=(1.0, 2.0, 4.0), every=1.0, history=1.0, **sources):
    # Mean absolute error of health and of distance, same start points as sections 10-12
    health, dist = {h: [] for h in horizons}, {h: [] for h in horizons}
    for i in ids:
        traj = trajectories[i]
        H, D = traj[STATE].to_numpy(), traj["distance"].ffill().bfill().to_numpy()
        for k0 in range(round(history / dt), len(H) - round(horizons[0] / dt), round(every / dt)):
            n_steps = min(round(horizons[-1] / dt), len(H) - 1 - k0)
            h, d = rollout(i, k0, n_steps, **sources)
            for x in horizons:
                n = round(x / dt)
                if n <= n_steps:
                    health[x].append(np.mean(np.abs(h[n] - H[k0 + n])))
                    dist[x].append(abs(d[n] - D[k0 + n]))
    return pd.concat(
        {
            "health error": pd.Series({x: np.mean(e) for x, e in health.items()}),
            "distance error": pd.Series({x: np.mean(e) for x, e in dist.items()}),
        }
    )


variants = {
    "all real (section 11)": dict(distance="data", depth="data", alive="data"),
    "all guessed (section 12 best)": dict(distance="trend", depth="trend", alive="health"),
    "real distance, depth held, alive from health": dict(distance="data", depth="hold", alive="health"),
    "trend distance, depth held, alive from health": dict(distance="trend", depth="hold", alive="health"),
    "real distance and alive, depth held": dict(distance="data", depth="hold", alive="data"),
    "MODEL: distance equation": dict(distance="model", depth="hold", alive="health"),
    **{
        f"MODEL: equation + closing speed, tau {tau} s": dict(
            distance="model + speed", depth="hold", alive="health", tau=tau
        )
        for tau in [0.25, 0.5, 1.0]
    },
}
horizon_table = pd.DataFrame({name: rollout_errors(test_ids, **v) for name, v in variants.items()}).T
horizon_table.round(3)

**Reading it** (health error, then distance error, at 4 s):

| | health | distance |
|---|---|---|
| all real (section 11) | 0.044 | 0 |
| real distance and alive, depth held | 0.043 | 0 |
| real distance, depth held, alive from health | 0.048 | 0 |
| trend distance, depth held, alive from health | 0.050 | 1.83 |
| all guessed (section 12 best) | 0.057 | 1.83 |
| **distance equation** | 0.055 | **0.41** |
| equation + closing speed, τ 0.25-1 s | 0.050-0.051 | 0.41-0.81 |

- **Depth needs no equation, at least over 4 s.** Held at its start value it is as good as the real
  depth (0.043 against 0.044). Extending its trend made section 12 worse: holding it alone takes
  the fair score from 0.057 to 0.050.
- **Alive from predicted health costs little** (0.048 against 0.043).
- **The distance equation predicts distance four times better than the trend** (0.41 against 1.83),
  but health slightly worse (0.055 against 0.050). Even the *real* distance only reaches 0.048:
  most start points are already in contact, where distance hardly changes.
- **Starting from the observed closing speed** (a second state $v$, relaxing towards the equation
  with time constant τ) brings health to 0.050-0.051. The current closing speed is part of the
  present state, not future data. Longer τ helps health and hurts distance.

### Whole battles, nothing read from the data

The real test of a world model: start 1 s into each test battle and run to the battle's end with
only the start state and Blue's setting. Scored by health error over the battle and at its end,
whether the winner is right, and the distance error.

In [ ]:
def whole_battle(i, **sources):
    traj = trajectories[i]
    H, D = traj[STATE].to_numpy(), traj["distance"].ffill().bfill().to_numpy()
    k0 = round(1.0 / dt)
    h, d = rollout(i, k0, len(H) - 1 - k0, **sources)
    return {
        "health error, mean": np.mean(np.abs(h - H[k0:])),
        "health error, end": np.mean(np.abs(h[-1] - H[-1])),
        "winner right": np.sign(h[-1, 0] - h[-1, 1]) == np.sign(H[-1, 0] - H[-1, 1]),
        "distance error": np.mean(np.abs(d - D[k0:])),
        "apart at start": D[k0] > 8,
    }


battle_variants = {
    "real distance": dict(distance="data"),
    "distance equation": dict(distance="model"),
    "equation + closing speed (tau 0.5 s)": dict(distance="model + speed", tau=0.5),
    "trend distance": dict(distance="trend"),
    "held distance": dict(distance="hold"),
}
battles = pd.concat(
    {name: pd.DataFrame([whole_battle(i, **v) for i in test_ids]) for name, v in battle_variants.items()},
    names=["distance from", "battle"],
)
winner_from_start = np.mean(
    [
        np.sign(trajectories[i]["health_blue"].iloc[8] - trajectories[i]["health_red"].iloc[8])
        == np.sign(trajectories[i]["health_blue"].iloc[-1] - trajectories[i]["health_red"].iloc[-1])
        for i in test_ids
    ]
)
print(f"winner = side ahead in health at 1 s: right in {winner_from_start:.1%} of test battles")
pd.concat(
    {
        "all test battles": battles.groupby(level=0).mean(),
        "apart at start (d > 8)": battles[battles["apart at start"]].groupby(level=0).mean(),
    }
).drop(columns="apart at start").round(3)

**Reading it:**

| distance from | health error, mean | at the end | winner right | distance error |
|---|---|---|---|---|
| real distance | 0.053 | 0.081 | 92.2% | 0 |
| **distance equation** | 0.059 | 0.086 | 92.0% | **0.29** |
| equation + closing speed (τ 0.5 s) | 0.053 | 0.070 | 92.4% | 0.76 |
| trend | 0.059 | 0.083 | 92.0% | 3.46 |
| held | 0.146 | 0.244 | 90.2% | 1.05 |

The side ahead in health at 1 s wins 90.4% of test battles.

- **The model now runs on its own,** from health, alive, distance and depth at one moment plus
  Blue's aggression, and stays within about 0.006 of the model fed the real distance. Holding
  distance fails badly (0.146): movement had to be modelled.
- **The distance equation is the only one that gets distance right** over whole battles (0.29,
  against 3.46 for the trend).
- **Worse distance, better health:** the closing-speed variant carries marching speed into contact
  and overshoots (distance error 0.76, 1.30 for battles that start apart), yet its health at the
  end beats even the real distance (0.070 against 0.081). That only works if something else
  **underestimates damage at first contact**: pushing the teams too close cancels it. A side check
  outside the notebook agrees: for start points at distance 7-12, the equation's health bias at 4 s
  is +0.015 to +0.033 (too little damage), and the trend's is −0.02 to −0.04.
- So the remaining error is **in contact, in the reach closure and in alive**, not in movement.
  Real reach gives 0.031 at 4 s; the closure with everything real gives 0.044.

**Winner accuracy barely moves** (90-92%): with these force ratios most battles are decided at
the start. It is a weak metric here.

## 14 · Reach in contact: alive, and Blue's retreat

Section 13 left the error in contact: even with the real distance, the closure gives 0.048 at 4 s
against 0.031 with real reach. Where is the closure wrong? Its bias (closure minus real reach) on
training battles, by time since first contact, and by Blue's retreat setting late in battles.
Inputs are real here: this is the closure's own error.

In [ ]:
def contact_rows(i):
    # Per step while both sides have units: real and closure reach, time since first contact, Blue's setting
    traj = trajectories[i]
    G = traj[GEOM].ffill().bfill().fillna(0).to_numpy().T
    closure = reach_from(G, traj[ALIVE].to_numpy().T)
    reach = traj[["reach_blue", "reach_red"]].to_numpy()
    t = traj["t"].to_numpy()
    first = t[reach.sum(axis=1) > 0].min() if (reach.sum(axis=1) > 0).any() else np.inf
    out = pd.DataFrame(
        {
            "bias_blue": closure[0] - reach[:, 0],
            "bias_red": closure[1] - reach[:, 1],
            "since_contact": t - first,
            "retreat": "team"
            if controls.loc[i, "team_below"] > 0
            else "unit"
            if controls.loc[i, "unit_below"] > 0
            else "none",
            "both_alive": (traj["alive_blue"].to_numpy() > 0) & (traj["alive_red"].to_numpy() > 0),
        }
    )
    return out[out["both_alive"]]


contact = pd.concat([contact_rows(i) for i in train_ids])
by_time = contact.groupby(pd.cut(contact["since_contact"], [-1, 0, 0.5, 1, 2, 4, 8, 30]), observed=True)
display(by_time[["bias_blue", "bias_red"]].mean().join(by_time.size().rename("rows")).round(3))
late = contact[contact["since_contact"] > 8].groupby("retreat")
late[["bias_blue", "bias_red"]].mean().join(late.size().rename("rows")).round(3)

**Reading it:**

- In the first second of contact the closure is **too low** by about 0.05 (a tenth of the reach
  then). Later it is about right, until **8 s after contact, where it is far too high** (0.03 / 0.09).
- Late in battles that excess is **almost all from battles where Blue retreats** (13% of battles,
  but most late rows, since they last long). Without retreat the late bias is −0.016 / −0.005.
- `RetreatPolicy`: a weak Blue unit moves away from enemies within 6 (just beyond the reach of
  5.75) and **doesn't shoot** while it does. The whole team retreats when Blue's health falls below
  `team_below` × Red's.

### Retreat as part of the model

Both triggers can be computed from the team state and Blue's setting. The **retreating share**
$\rho$:

- **team retreat:** $\rho = 1$ when $H_B < \text{team\_below} \cdot H_R$
- **unit retreat:** a smooth step in how far the mean Blue unit's health $H_B / A_B$ is below
  `unit_below`: $\rho = 1/(1 + e^{(H_B/A_B - \text{unit\_below} - m_0)/w_m})$. Some units fall below the threshold before the mean does.

Three places it could enter, each fitted on training battles:

1. **reach:** $R = A\,s(d, \sigma)\,(1 - q\,\rho)$ per side: retreating units keep out of reach
2. **firepower:** Red's losses $\dot H_R = -k_R\,R_B\,(1 - s_f\,\rho)$: units in reach that retreat don't fire
3. **movement:** $\dot d = -v\,g(d)\,(1 - \rho)$: a retreating Blue stops closing

And a fourth change, needed for the unit trigger: with alive scaled by health (section 12), the
mean unit health never changes, so the trigger never fires. Damage is spread over units (F6), so
alive falls more slowly than health: $A = A_0 (H/H_0)^p$ with $p < 1$.

In [ ]:
def retreat_share(hb, hr, ab, unit_below, team_below, m0, wm):
    # Share of Blue retreating, from team health, Blue's alive and Blue's setting (works on arrays)
    with np.errstate(all="ignore"):
        unit = np.where(unit_below > 0, 1 / (1 + np.exp((hb / ab - unit_below - m0) / wm)), 0.0)
    return np.where((team_below > 0) & (hb < team_below * hr), 1.0, unit)


setting = controls.loc[np.repeat(train_ids, [len(trajectories[i]) for i in train_ids])]
fit_rows = rows.assign(
    unit_below=setting["unit_below"].to_numpy(),
    team_below=setting["team_below"].to_numpy(),
    dH_blue=DX[:, 0],
    dH_red=DX[:, 1],
)
fit_rows = fit_rows[(fit_rows["alive_blue"] > 0) & (fit_rows["alive_red"] > 0)]
closure_train = reach_from(fit_rows[GEOM].fillna(0).to_numpy().T, fit_rows[ALIVE].to_numpy().T)
hb, hr, ab, ub, tb = (
    fit_rows[c].to_numpy() for c in ["health_blue", "health_red", "alive_blue", "unit_below", "team_below"]
)


# 1 · reach
def reach_with_retreat(X, q_blue, q_red, m0, wm):
    rho = retreat_share(*X, m0, wm)
    return np.concatenate([closure_train[0] * (1 - q_blue * rho), closure_train[1] * (1 - q_red * rho)])


real_reach = np.concatenate([fit_rows["reach_blue"], fit_rows["reach_red"]])
(q_blue, q_red, m0, wm), _ = curve_fit(
    reach_with_retreat, (hb, hr, ab, ub, tb), real_reach, p0=(0.4, 0.6, -0.1, 0.05)
)
r2_closure = 1 - np.sum((real_reach - np.concatenate(closure_train)) ** 2) / np.sum(
    (real_reach - real_reach.mean()) ** 2
)
r2_retreat = 1 - np.sum(
    (real_reach - reach_with_retreat((hb, hr, ab, ub, tb), q_blue, q_red, m0, wm)) ** 2
) / np.sum((real_reach - real_reach.mean()) ** 2)
print(
    f"reach: q_blue = {q_blue:.2f}, q_red = {q_red:.2f}, m0 = {m0:.3f}, wm = {wm:.3f}; "
    f"R² {r2_closure:.2f} -> {r2_retreat:.2f}"
)

# 2 · firepower: Red's losses on Blue's reach, and Blue's reach times rho
rho_train = retreat_share(hb, hr, ab, ub, tb, m0, wm)
(k_fire, k_fire_rho), *_ = np.linalg.lstsq(
    np.column_stack([fit_rows["reach_blue"], fit_rows["reach_blue"] * rho_train]),
    fit_rows["dH_red"],
    rcond=None,
)
s_fire = -k_fire_rho / k_fire
print(f"Red's losses = {k_fire:.3f} reach_blue + {k_fire_rho:.3f} reach_blue * rho  ->  s_f = {s_fire:.2f}")
by_rho = fit_rows.assign(rho=rho_train).groupby(pd.cut(rho_train, [-0.01, 0.01, 0.5, 0.99, 1]), observed=True)
display(
    by_rho.apply(lambda g: -np.dot(g["reach_blue"], g["dH_red"]) / np.dot(g["reach_blue"], g["reach_blue"]))
    .rename("Red's losses per Blue unit in reach")
    .round(3)
)

# 4 · alive against health, per battle from its start
log_h, log_a = [], []
for i in train_ids:
    for side in ["blue", "red"]:
        h, a = trajectories[i][f"health_{side}"].to_numpy(), trajectories[i][f"alive_{side}"].to_numpy()
        ok = (h > 0) & (a > 0)
        log_h.append(np.log(h[ok] / h[0]))
        log_a.append(np.log(a[ok] / a[0]))
log_h, log_a = np.concatenate(log_h), np.concatenate(log_a)
p_alive = np.dot(log_h, log_a) / np.dot(log_h, log_h)
print(f"alive = A0 (H/H0)^p, p = {p_alive:.2f}")

**Reading it:**

- **Reach:** $q_B$ = 0.39, $q_R$ = 0.77, R² of reach from 0.82 to **0.92**. When Blue retreats, Red
  loses three quarters of its units in reach, and Blue loses about two fifths: retreating units
  keep out of reach, Red's units chase. The unit trigger switches when the mean unit is about
  0.15 below `unit_below` ($m_0$ = −0.146, a sharp step).
- **Firepower:** $s_f$ = **0.81**. Per Blue unit in reach, Red loses 0.186 with no retreat and about
  0.04 during a full retreat. That is the policy's "a retreating unit doesn't shoot", recovered
  from team-level data.
- **Alive:** $p$ = 0.58. Alive falls much more slowly than health: damage is spread over units, so
  units die late (F6).

### Predicting with retreat

`rollout_v2` is section 13's `rollout` with the four changes as switches. Distance comes from the
equation, depth is held, alive follows predicted health. Each row adds one change.

In [ ]:
def rollout_v2(
    i,
    k0,
    n_steps,
    distance="model",
    reach="closure",
    alive_p=1.0,
    reach_retreat=False,
    fire=False,
    stop=False,
):
    # Health and distance over n_steps from step k0, nothing read from the future (unless distance
    # or reach is "data"). The switches add Blue's retreat to reach, firepower and movement.
    traj = trajectories[i]
    H, A = traj[STATE].to_numpy(), traj[ALIVE].to_numpy()
    G = traj[GEOM].ffill().bfill().fillna(0).to_numpy()
    R = traj[["reach_blue", "reach_red"]].to_numpy()
    timid, unit_below, team_below = (
        1 - controls.loc[i, "aggression"],
        controls.loc[i, "unit_below"],
        controls.loc[i, "team_below"],
    )
    inputs = np.zeros((1, len(DISTANCE_FEATURES) - 1))
    h, d = H[k0].copy(), G[k0, 0]
    healths, distances = [h.copy()], [d]
    for n in range(1, n_steps + 1):
        k = k0 + n - 1
        a = A[k0] * np.divide(h, H[k0], out=np.zeros(2), where=H[k0] > 0) ** alive_p
        rho = float(retreat_share(h[0], h[1], a[0], unit_below, team_below, m0, wm)) if a[0] > 0 else 0.0
        if reach == "data":
            r = R[k]
        else:
            r = reach_from(np.array([G[k, 0] if distance == "data" else d, G[k0, 1], G[k0, 2]]), a)
            if reach_retreat:
                r = r * np.array([1 - q_blue * rho, 1 - q_red * rho])
        red_fire = 1 - s_fire * rho if fire else 1.0
        h = np.clip(h + dt * np.array([-k_reach["blue"] * r[1], -k_reach["red"] * r[0] * red_fire]), 0, None)
        g = closing_gate((d, G[k0, 1], G[k0, 2]), 1.0, *gate_shape)
        inputs[0, 2:4] = g, timid * g
        v_star = distance_model.predict(np.array([[d]]), u=inputs)[0, 0]
        d = max(d + dt * v_star * ((1 - rho) if stop else 1), 0)
        healths.append(h.copy())
        distances.append(G[k0 + n, 0] if distance == "data" else d)
    return np.array(healths), np.array(distances)


def score_v2(**options):
    # 4 s horizon errors (same start points as sections 10-13) and whole battles from 1 s in
    horizons = {x: [] for x in (1.0, 2.0, 4.0)}
    battles = []
    for i in test_ids:
        H = trajectories[i][STATE].to_numpy()
        for k0 in range(round(1.0 / dt), len(H) - round(1.0 / dt), round(1.0 / dt)):
            n_steps = min(round(4.0 / dt), len(H) - 1 - k0)
            h, _ = rollout_v2(i, k0, n_steps, **options)
            for x in horizons:
                if round(x / dt) <= n_steps:
                    horizons[x].append(np.mean(np.abs(h[round(x / dt)] - H[k0 + round(x / dt)])))
        k0 = round(1.0 / dt)
        h, _ = rollout_v2(i, k0, len(H) - 1 - k0, **options)
        battles.append(
            {
                "battle, mean": np.mean(np.abs(h - H[k0:])),
                "battle, end": np.mean(np.abs(h[-1] - H[-1])),
                "winner right": np.sign(h[-1, 0] - h[-1, 1]) == np.sign(H[-1, 0] - H[-1, 1]),
                "retreat battle": controls.loc[i, "unit_below"] > 0 or controls.loc[i, "team_below"] > 0,
            }
        )
    battles = pd.DataFrame(battles)
    return pd.concat(
        [
            pd.Series({f"{x:g} s": np.mean(e) for x, e in horizons.items()}),
            battles.drop(columns="retreat battle").mean(),
            battles[battles["retreat battle"]]
            .drop(columns="retreat battle")
            .mean()
            .add_prefix("retreat battles: "),
        ]
    )


steps = {
    "section 13 model": {},
    "+ alive from health^p": dict(alive_p=p_alive),
    "+ retreating units don't fire": dict(alive_p=p_alive, fire=True),
    "+ retreat stops closing": dict(alive_p=p_alive, fire=True, stop=True),
    "+ retreat lowers reach (full model)": dict(alive_p=p_alive, fire=True, stop=True, reach_retreat=True),
    "full model, real distance": dict(alive_p=p_alive, fire=True, reach_retreat=True, distance="data"),
    "real reach (oracle, section 9)": dict(reach="data"),
    "real reach + alive^p + fire": dict(reach="data", alive_p=p_alive, fire=True),
}
retreat_table = pd.DataFrame({name: score_v2(**options) for name, options in steps.items()}).T
retreat_table.round(3)

**Reading it** (health error; nothing read from the future except in the last three rows):

| | 4 s | whole battle, mean | at the end | retreat battles, mean |
|---|---|---|---|---|
| section 13 model | 0.055 | 0.059 | 0.086 | 0.073 |
| + alive from health^p | 0.049 | 0.050 | 0.057 | 0.069 |
| + retreating units don't fire | 0.044 | 0.048 | 0.056 | 0.058 |
| + retreat stops closing | 0.042 | 0.048 | 0.056 | 0.059 |
| **+ retreat lowers reach (full model)** | **0.040** | 0.049 | 0.056 | 0.062 |
| full model, real distance | 0.036 | 0.043 | 0.053 | 0.060 |
| real reach (oracle, section 9) | 0.031 | 0.039 | 0.045 | 0.061 |
| real reach + alive^p + fire | 0.027 | 0.037 | 0.042 | 0.045 |

- **The full model predicts better than section 11 did with the real future geometry** (0.040
  against 0.044 at 4 s), with nothing read from the future. Against the health-only model
  (0.065), it closes about three quarters of the gap to the old oracle (0.031).
- **Alive^p is the biggest single gain**, mostly over whole battles (end error 0.086 → 0.057).
  Scaling alive with health killed units too early, and with them their firepower.
- **"Retreating units don't fire" is the biggest gain in retreat battles** (0.069 → 0.058). It
  also lowers the oracle: with the real reach, 0.031 → 0.027 at 4 s.
- **Retreat in reach and movement helps at 4 s but not over whole battles** (0.048 → 0.049). Over
  long rollouts, errors in the unit trigger add up. A judgement call; the full model is kept
  because every term has a physical reading and a fitted size.
- **Winner accuracy doesn't move** (92%). With real reach it is 97.6%: what decides close battles
  is still in reach, not in the terms here.
- Left: the closure's 10% shortfall in the first second of contact (not addressed here), and
  the trigger relies on Blue's `unit_below` and `team_below` being known. They are Blue's own
  policy, so a planner knows them.

## Next

What the notebook shows, in order of strength:

1. **Losses follow the opponent's units within reach** (section 9): STLSQ picks
   `(health_blue)' = -0.176 reach_red`, `(health_red)' = -0.183 reach_blue` out of health, alive and
   reach. R² 0.72-0.73 on the loss rates, better than the measured engaged count.
2. **Knowing reach gives the best prediction** (health error 0.031 at 4 s, against 0.065 for the
   best health-only model).
3. **Reach follows from distance and formation depth** (section 11): a sigmoid whose switch lands
   on the per-unit reach (5.76 / 5.74 against 5.75), shifted by the enemy's depth and widened by
   one's own. R² 0.74 (Red) and 0.92 (Blue); health error 0.044 at 4 s, about 60% of the way from
   the health-only model to the oracle.
4. **With no future data it still wins** (section 12): trend geometry and alive from predicted health
   give 0.057 at 4 s, against 0.065 for the health-only model.
5. **Distance has an equation, and the model runs on its own** (section 13): SINDy keeps
   `(distance)' = -5.78 gate + 2.18 timid*gate`, closing at marching speed until contact, slower
   for a less aggressive Blue. Depth can be held, alive follows predicted health. Whole battles from
   1 s in: health error 0.059 (0.053 with the real distance). The remaining error is in contact.
6. **Blue's retreat and spread damage close most of the rest** (section 14): alive falls as
   health^0.58, retreating units don't fire ($s_f$ = 0.81, recovered from team data) and keep out of
   reach. Health error 0.040 at 4 s with nothing read from the future, better than section 11 with
   the real geometry (0.044).
7. **An equation for reach** (section 10, movement plus deaths) is stable and physical, but adds
   nothing over the closure beyond 2 s.

So the team-level state that carries the dynamics is **[health, alive, distance, depth]** per
side, with reach computed from it and health losses driven by reach. Since sections 13-14 nothing
is read from the future: distance has an equation, depth is held, alive follows health^0.58, and
Blue's setting (aggression, retreat thresholds) enters as a control.

Next steps:

- ~~**Close the loop**~~ done in section 13. Left: the reach closure's shortfall in the first second
  of contact (section 14), and movement intent that changes within battles (Q1 in `docs/findings.md`).
- **Unexplained:** the in-range drop at small distances (section 9, target choice), movement
  coefficients below 1 (section 10), Blue's long depth tail.
- **Code:** move `geometry`, `reach_features` and `in_reach_share` into `src/sindy_wm/` with tests.